# HW06: Attention

Remember that these homework work as a completion grade. **You can <span style="color:red">not</span> skip one section this homework.**

### Data sampling with a sliding window

In this intro code I will ask you to revise what we saw on the lab to test your understanding of the features.

In [ ]:
#Import the AG news dataset (same as hw01)
#Download them from here 
# !wget https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/train.csv

import pandas as pd
import nltk
df = pd.read_csv('train.csv')

df.columns = ["label", "title", "lead"]
label_map = {1:"world", 2:"sport", 3:"business", 4:"sci/tech"}
def replace_label(x):
	return label_map[x]
df["label"] = df["label"].apply(replace_label) 
df["text"] = df["title"] + " " + df["lead"]
df = df.sample(n=10000) # # only use 10K datapoints
df.head()

By now you should have a clear understanding of how this data is conformed. 

In [ ]:
#TODO clean up the text as best as you can. Build a function to do this and generate a new variable called 'text_clean'

In [ ]:
#TODO using byte-pair encoding, tokenize the first article of the dataset. select 10 tokens that make a coherent sentence
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")


#TODO For this 10 tokens, use the sliding window DataLoader seen in the lab to generate the input and target tensors:

from torch.utils.data import Dataset, DataLoader
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt) # Tokenizes the entire text

        for i in range(0, len(token_ids) - max_length, stride):  # Uses a sliding window to chunk the book into overlapping sequences of max_length
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self): # Returns the total number of rows in the dataset
        return len(self.input_ids)

    def __getitem__(self, idx):  # Returns a single row from the dataset
        return self.input_ids[idx], self.target_ids[idx]

# Now A data loader to generate batches with input-with pairs
def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2") # Initializes the tokenizer
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride) # Creates dataset
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last, # drop_last=True drops the last batch if it is shorter than the specified batch_size to prevent loss spikes during training. (See Rashkca A6)
        num_workers=num_workers  # The number of CPU processes to use for preprocessing
    )

    return dataloader

In [ ]:
#TODO generate at least 3 different combinations of input-target tensors using different strides and max_lengths

#TODO for each combination, print the output in words (decode) and explain what's happening


### Word and positional embeddings

In [ ]:
#TODO Using the input tokens, create a token embedding layer and look up the embeddings

#TODO create a positional embedding layer

#TODO generate position indices [0, 1, 2] and look them up

#TODO Sum them up and interpret the result (.shape, what are the values?)

### Fine-tuning BERT

In [ ]:
# create a new variable "business" that takes value 1 if the label is business and 0 otherwise

df['business'] = df['label'].apply(lambda x: int(x=='business'))
y = df['business'].values
df['business'].head()

#TODO fine-tune a small bert for predicting the business variable. Plot the loss and accuracy by epoch (test and train)

import torch
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm

# ── 1. Data (already preprocessed from your snippet) ──────────────────────────
df_model = df[['text', 'business']].dropna()
df_model['business'] = df_model['business'].astype(int)

train_df, val_df = train_test_split(df_model, test_size=0.2, random_state=42,
                                     stratify=df_model['business'])

# ── 2. Dataset ─────────────────────────────────────────────────────────────────
class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts      = texts.tolist()
        self.labels     = labels.tolist()
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }

# ── 3. Model & tokenizer ───────────────────────────────────────────────────────
MODEL_NAME = "prajjwal1/bert-tiny"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)



In [ ]:
#TODO add a dropout and a ReLU: [CLS] token (128) → Linear → ReLU → Dropout(0.3) → Linear → 2 logits